## Heroes of Gold and Move Points

Welcome to a Heroes-themed VRPTW educational challenge

Source of this challenge is Data Fusion 2026 competition on ods.ai:

https://ods.ai/competitions/data-fusion2026-heroes

Recap of the problem:
- Our goal is to construct a route across Gold Waterwheel Mills (objects with `object_id`) for a fleet of our heroes (like couriers in VRP, with `hero_id`) 
- All heroes start from the same Castle (Depot) and are hired sequentially. Unlike HoMM we are allowed to hire as many heroes at once as we please
- Objects appear on the map at different Days: arriving late yields no reward and arriving early invokes VRPTW implicit waiting convention (`day_open` is equivalent to time windows)
- Primary task: route should maximise Gold Score: collected gold `reward` of all successfully visited objects (-) cost of hiring all heroes

For complete and detailed set of Rules and conventions check ods.ai: 

https://ods.ai/competitions/data-fusion2026-heroes/problem

For this educational task we provide a `heroes_utils.py` with utility tools which will help us solve it

In [1]:
import polars as pl
from heroes_utils import HeroesInstance

Our main tool for this task is `HeroesInstance` class:
- At init it loads information about heroes and objects, as well as distances for both pairwise object distances and distance to each object from Castle/Depot. You can access them via `heroes`, `objects`, `dist_matrix` and `dist_start`
- Under the hood HeroesInstance also provides utility lookups `hero_mp_map`, `obj_info_map` and `dist_start_map`

Key functions of `HeroesInstance` class:
- Main function of our instance is to get a full evaluation of a hero routing solution `evaluate_solution` which produces our *Gold Score*
- In order to get there we need 2 extra functions: `basic_check` to validate our solution candidate and `expand_solution`. Solution expansion function is the core of the whole evaluation as it thoroughly simulates routes w.r.t. all our rules and conventions
- Inner workings of hero routing simulation can be found in `hero_journey` function (iterative simulation for a given hero) and `simulate_hero_movement`. Please refer to the latter for all the logic and mechanics of each hero transition (w.r.t. their current state at the time of transition).
- Please note that you can use `expand_solution` in an unsafe manner without all the steps of a `basic_check`

Let's now create this instance and load our data. Original data for this problem is available on ods.ai:

https://ods.ai/competitions/data-fusion2026-heroes/dataset

In [2]:
instance = HeroesInstance('data/')

Great, now we have our instance with all of our data operational!

Now let's make a super basic route: let's take our first hero and make him walk across objects 1..5.

We also convert this route to our expected submit format which we have to get used to:

In [3]:
route_dummy_basic = {
    'hero_id': [1] * 5,
    'object_id': [1, 2, 3, 4, 5]
}
submit_dummy_basic = pl.DataFrame(route_dummy_basic)
submit_dummy_basic

hero_id,object_id
i64,i64
1,1
1,2
1,3
1,4
1,5


Now let's take a look at how this potential route submit will look like if we expand it

In [4]:
detailed_dummy_basic = instance.expand_solution(submit_dummy_basic)
detailed_dummy_basic

hero_id,object_id_from,object_id_to,day_start,day_arrive,day_leave,move_points_start,move_points_arrive,move_points_burned,move_points_leave,is_earlier,is_late,reward
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,bool,bool,i64
1,0,1,1,1,1,1700,1287,0,1187,false,false,500
1,1,2,1,1,4,1187,887,4287,1600,true,false,500
1,2,3,4,4,4,1600,1393,0,1293,false,false,500
1,3,4,4,4,4,1293,853,0,753,false,true,0
1,4,5,4,4,4,753,206,0,106,false,true,0


This detailed report-like DataFrame is essential to understand what is actually going on in this challenge

Let's work though it step by step:
1. Our first object is (1)
    - It means that our first transition is 0 -> 1 (start at Castle/Depot) 
    - We had 1700 move points at start, spent 513 on movement and successfully arrived there
    - This arrival was on-time (neither early nor late)
    - At arrival we were charged a `VISIT_COST` of 100 and generated our 500 gold in reward
2. Now moving to our next object (2): 
    - Our next destination is (2): 1 -> 2 
    - We arrived there at Day 1, however this object only opens on Day 4
    - This means that our hero arrived early and had to wait there for 2.5 days hence he got 4287 move points burned (1700 x 2 + 887 when he arrived)
    - Then at the start of Day 4 we were once again charged 100 for a visit, generated 500 gold reward and moved on
3. Luck is back on our side with object (3):
    - Next transition 2 -> 3 has object (3) available and open on Day 4
    - Hence the logic is straightforward and works just like our first (starting) trip
    - We spend 207 move points on this transit, get charged with extra 100 for visit and procced onwards with another 500 gold
4. New problems arrive with object (4):
    - Our next transit is to (4): 3 -> 4
    - Unfortunately object (4) was available on Day 3, but our hero arrives a day later on Day 4 
    - We are still charged 440 move points for travelling there, as well as a visit cost of 100 
    - Unfortunately there is no gold reward available and the hero simply moves with 0 reward
5. More problems at our last object (5):
    - During our last transition 4 -> 5 we run into the same late arrival problem
    - Object (5) was open on Day 1 however at current state of the route our hero is in the middle of his Day 4 path
    - Hence it is another failure with a late visit
    - As this was the last object on our route, we consider that our hero can now be discharged 

In total, during this route our hero succesfully collected 1500 gold. As hiring one hero cost us 2500, our total Gold Score should be -1000 in gold

In [5]:
instance.evaluate_solution(submit_dummy_basic)

-1000

Let's now craft a different schedule with some broken and unrealistic details.

Imagine that in our routing solutions some things broke in several places: 
- We somehow got extra heroes (102) and extra objects (705, 725, 745)
- We also managed to duplicate our rows (hero 1, object 2)
- And also we somehow managed to have the same object be visited by several heroes (object 3)

In [6]:
route_dummy_problematic = {
    'hero_id': [1, 1, 1, 1, 2, 2, 2, 102, 102, 102],
    'object_id': [1, 2, 2, 3, 3, 4, 5, 705, 725, 745]
}

submit_dummy_problematic = pl.DataFrame(route_dummy_problematic)
submit_dummy_problematic

hero_id,object_id
i64,i64
1,1
1,2
1,2
1,3
2,3
2,4
2,5
102,705
102,725


If we perform a `basic_check` on this submit candidate, only allowed rows would remain for evaluation

In [7]:
instance.basic_check(submit_dummy_problematic)

hero_id,object_id
i32,i32
1,1
1,2
1,3
2,4
2,5


However, as we previously noted, `expand_solution` function can be used in unsafe manner and thus bring several unrealistic transits into your overview

So either be careful or adjust your functions accordingly :)

In [8]:
detailed_dummy_problematic = instance.expand_solution(submit_dummy_problematic)
detailed_dummy_problematic

hero_id,object_id_from,object_id_to,day_start,day_arrive,day_leave,move_points_start,move_points_arrive,move_points_burned,move_points_leave,is_earlier,is_late,reward
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,bool,bool,i64
1,0,1,1,1,1,1700,1287,0,1187,false,false,500
1,1,2,1,1,4,1187,887,4287,1600,true,false,500
1,2,2,4,4,4,1600,1600,0,1500,false,false,500
1,2,3,4,4,4,1500,1293,0,1193,false,false,500
2,0,3,4,4,4,1560,1174,0,1074,false,false,500
2,3,4,4,4,4,1074,634,0,534,false,true,0
2,4,5,4,5,5,534,1547,0,1447,false,true,0


As you can see, we have only 5 legit transits after `basic_check` yet 7 transits after `expand_solution`

If we look closer, instead of 5 reward-producting objects we would only get 4 (corresponding to basic_check results; abusing repeated visits with 0 move points spent is prohibited)

Thus our gold score should be -3000: 2 heroes costs 5000 and 4 objects can bring us 2000 gold

In [9]:
instance.evaluate_solution(submit_dummy_problematic)

-3000

It might also be helpful to use option `remove_out_of_time = True` for your expanded solutions. 

Imagine we tried an ultra-naive TSP-like solution baseline: 
- We take our first hero and no one else
- We make him visit all our objects one by one

Let's make this ultra-naive submit example:

In [10]:
route_dummy_ultranaive = {
    'hero_id': [1] * 20,
    'object_id': list(range(1, 21))
}

submit_dummy_ultranaive = pl.DataFrame(route_dummy_ultranaive)
submit_dummy_ultranaive

hero_id,object_id
i64,i64
1,1
1,2
1,3
1,4
1,5
…,…
1,16
1,17
1,18


Due to Day-based time windows, soon enough all objects will become unfeasible:
- Mechanically they are reachable in terms of move points
- However our hero will arrive there at Day 8 or later

As our game ends after Day 7, it will be useful to look only at feasible transits in our hero routes

In [11]:
# Mind that although we could travel to previous objects, most of them are OOT (Day 8 and beyond)
detailed_dummy_ultranaive = instance.expand_solution(submit_dummy_ultranaive, remove_out_of_time = True)
detailed_dummy_ultranaive

hero_id,object_id_from,object_id_to,day_start,day_arrive,day_leave,move_points_start,move_points_arrive,move_points_burned,move_points_leave,is_earlier,is_late,reward
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,bool,bool,i64
1,0,1,1,1,1,1700,1287,0,1187,false,false,500
1,1,2,1,1,4,1187,887,4287,1600,true,false,500
1,2,3,4,4,4,1600,1393,0,1293,false,false,500
1,3,4,4,4,4,1293,853,0,753,false,true,0
1,4,5,4,4,4,753,206,0,106,false,true,0
1,5,6,4,5,7,106,1693,3393,1600,true,false,500
1,6,7,7,7,7,1600,1080,0,980,false,true,0
1,7,8,7,7,7,980,540,0,440,false,true,0


As you can see, we didn't have to put all our 700 objects into a route

Only 8 objects were reachable with this simple route during our 7 Day game week. However this doesn't imply that a single hero can cover only 7 objects 

In [12]:
instance.evaluate_solution(submit_dummy_ultranaive)

-500

Let's now craft a baseline with a simple greedy (and lazy) heuristic

### Background ideas: 
- Let's keep track of all available (unassigned) objects
- Don't be afraid to loop over all available objects at any time with `simulate_hero_movement` 
- Choose the greediest and laziest option at each step

### Baseline flow — greedy and lazy:
- Iterate over heroes (starting from `hero_id == 1`)
- Init hero's state with the closest available object to Castle/Depot
- Try (iterate) all available objects
- Choose the one which requires minimal move points and yet grants a reward
- ~~While choosing be careful to keep track of days and choose either the same-day-option, or the nearest day~~
- Stop hero greed when there are no available rewards left
- Stop global greed when there are either no more available objects (likely) or no more heroes left

In [13]:
def generate_greedy_solution(instance: HeroesInstance) -> pl.DataFrame:
    """
    Generate a greedy and lazy baseline solution 
    """

    # Prep work        
    available_ids = set(instance.objects['object_id'].to_list())
    solution_rows = []
    
    # Helper to get object info easily
    obj_info_map = instance.obj_info_map
    
    # Go through each hero in hiring (sequential) order
    for hero_id in range(1, 101):
        if not available_ids:
            break
            
        # Init first objects. We expect them to be from Day 1
        init_candidate_ids = [object_id for object_id in available_ids if obj_info_map[object_id]['day_open'] == 1]
        
        # Fallback in case there are no more Day 1 objects left (choose the nearest day)
        if not init_candidate_ids:
            # Added check for empty available_ids just in case
            if not available_ids: break
            min_day = min([obj_info_map[object_id]['day_open'] for object_id in available_ids])
            init_candidate_ids = [object_id for object_id in available_ids if obj_info_map[object_id]['day_open'] == min_day]
        
        if not init_candidate_ids:
            continue
            
        # Choose the nearest object to the Castle/Depot 
        best_start_id = min(init_candidate_ids, key = lambda object_id: instance.get_distance(0, object_id))
        
        # Reuse our state-switching logic from heroes_utils
        current_state = {
            'current_object': 0, 
            'current_day': 1, 
            'current_move_points': instance.hero_mp_map.get(hero_id, 0)
        }
        
        # Process our first hero transit
        transit = instance.simulate_hero_movement(hero_id, current_state, best_start_id)
        
        # Track our first move: add to submit, remove from available objects
        solution_rows.append({'hero_id': hero_id, 'object_id': best_start_id})
        available_ids.remove(best_start_id)
        
        # Update current state 
        current_state = {
            'current_object': transit['object_id_to'],
            'current_day': transit['day_leave'],
            'current_move_points': transit['move_points_leave']
        }
        
        # Improvise, adapt, iterate 
        while available_ids:
            best_transit = None
            transit_candidates = []
            
            # Greedy look at all remaining objects (at worst O(N^2)? or not?)
            for object_id in available_ids:
                try_new_object = instance.simulate_hero_movement(hero_id, current_state, object_id)

                if try_new_object['reward'] > 0:
                    transit_candidates.append(try_new_object)
            
            # No more reward-granting objects left, move on to next hero_id
            if not transit_candidates:
                break
                
            # Maybe we need better logic, especially with Day tracking
            # For now let's minimise the total move points of transitions — be greedy and lazy
            best_transit = min(transit_candidates, key = lambda x: x['move_points_arrive'] - x['move_points_start'] + x['move_points_burned'])
            
            # Proceed with our choice, add best transit option to route, remove from available ids
            chosen_id = best_transit['object_id_to']
            solution_rows.append({'hero_id': hero_id, 'object_id': chosen_id})
            available_ids.remove(chosen_id)
            
            # Update state, proceed to next greedy object transit
            current_state = {
                'current_object': best_transit['object_id_to'],
                'current_day': best_transit['day_leave'],
                'current_move_points': best_transit['move_points_leave']
            }

    if not solution_rows:
        return pl.DataFrame(schema={'hero_id': pl.Int32, 'object_id': pl.Int32})
        
    return pl.DataFrame(solution_rows)

One could use some better sophisticated logic even for this greedy baseline, but we should leave it as an excersize :)

Let's run it and see if it works

In [14]:
submit_lazy_greedy = generate_greedy_solution(instance)
submit_lazy_greedy

hero_id,object_id
i64,i64
1,427
1,514
1,1
1,482
1,580
…,…
46,254
46,412
46,555


Looks promising

Even the simplest heuristic managed to cover all 700 objects in just 47 heroes

Let's take a quick look at our detailed routing information

In [15]:
detailed_lazy_greedy = instance.expand_solution(submit_lazy_greedy)
detailed_lazy_greedy

hero_id,object_id_from,object_id_to,day_start,day_arrive,day_leave,move_points_start,move_points_arrive,move_points_burned,move_points_leave,is_earlier,is_late,reward
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,bool,bool,i64
1,0,427,1,1,1,1700,1594,0,1494,false,false,500
1,427,514,1,1,1,1494,614,0,514,false,false,500
1,514,1,1,1,1,514,1,0,0,false,false,500
1,1,482,1,2,2,0,1000,0,900,false,false,500
1,482,580,2,2,3,900,0,0,1600,true,false,500
…,…,…,…,…,…,…,…,…,…,…,…,…
46,0,254,1,1,1,1560,1134,0,1034,false,false,500
46,254,412,1,1,1,1034,841,0,741,false,false,500
46,412,555,1,1,1,741,514,0,414,false,false,500


As we can see, we now get special cases of Last Move Rule:
- Hero (1) wants to make a transit 514 -> 1
- After travelling there our hero has only 1 move point left, which is not enough to pay 100 `VISIT_COST` for a visit
- Due to this special rule mechanic we consider this visit was a success and still count it as valid
- In other words, our hero saved those missing 99 points during his travel
- As expected, our hero starts his next day with fully replenished move points 

Now let's see how much gold we get with this simple heuristic

In [16]:
instance.evaluate_solution(submit_lazy_greedy)

232500

Not bad for a baseline

Let's immediately save it as `sample_submit.csv` and send it to ods.ai to appear on the leaderboard

In [17]:
submit_lazy_greedy.write_csv("sample_submit.csv")

### Considerations for further solution improvements

- The simplest ideas to implement and get 283000+ are on the surface and can be achieved with 3 more lines of code. Expected limit of a greedy approach here is around 290000
- In real world, one can consider trying more sophisticated optimization techniques. In 2022 there was even a NeurIPS competition on VRPTW with winning solutions published (mostly genetic algorithm based; there were also academy-driven VRPTW competitions is 2021, 2023 and 2024)
- In other facets of a real world, you could try to use production-ready solvers. Good luck with adjusting them to our move point and Day-based logic though :)
- Coding LLMs can be helpful, but please try to use them consciously and as a coding-force multiplier rather than succumbing to a mere interface between LLM and a leaderboard. First of all this challenge is considered educational for humans


### Considerations for getting Companion nomination prizes

Just in case you want to receive a competition prize, please remeber that prizes are granted for public solutions only. Competition leaderboard is expected to saturate and thus has no implications on who gets a prize based on Gold Score

In order to receive a public solution prize, just as in Heroes of Might and Magic you can focus on different Heroes-themed paths:

- Might heroes: making public solutions on coding and maths, for exmaple on hardcore optimization. Try implementing custom logic of genetic algorithms for a Heroes-themed VRPTW problem
- Magic heroes: solving this problem in a different setting. For example expand the HeroesInstance to track number of heroes available at each day and make a solution which has 8+1 heroes just like in original Heroes game (and explicitly discharge heroes). Or solve this problem under Capacitated setting with no more than N objects for each hero (or N objects per day per hero with a max cap across all days). You can even reimplement it into a TW-Warhammer-style setting with extra costs on maintaining a hero at the start of each day
- Support heroes: make EDA and visualizations for objects, routing solutions and other fancy and helpful utilities (like score tracking across days, hero/courier metrics across days etc). Or simulations on metric and solver behaviour under different `_COST` and `reward` parameters 
- ...
- And random mixes of any of the above. For example solving a harder version under uniformity of hero routing schedules metric would likely require extra visualizations. Same goes for solutions aimed at optimizing different metrics (like classic VRPTW with total movement component)

Best of luck!